# H6 · Inside the EU: do higher labour costs go with higher productivity?

> **Hypothesis H6:** Within the EU, countries with higher hourly labour costs also have higher productivity (value added per employee), so expensive labour is not simply a burden on business.

| | |
|---|---|
| **Why we ask** | Eurostat gives EU-only measures the world-wide sources lack: what an hour of labour costs, what an employee produces, minimum wages, business profitability. The first check is whether labour cost tracks productivity; the second (exploratory) is what that means for profit. |
| **Prediction** | **Positive** association between hourly labour cost and value added per employee. |
| **Data** | Eurostat, 21 EU countries: hourly labour cost (business economy, euro, 2020–2024), value added per employee (2021–2024), gross operating rate, profit share and return on equity of non-financial corporations, minimum wage in purchasing power, unemployment. Yahoo country medians for firm outcomes. |
| **Primary test (fixed before looking at the result)** | **Spearman ρ between hourly labour cost and value added per employee** across the EU countries (bootstrap interval; p-value from the Spearman test). |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. *H6 is deliberately a sanity anchor: the primary link is expected to be strong. The informative part is the exploratory section on profit, which is corrected for multiple testing within the notebook and not counted as the primary test.* |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra=""):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy), outcome=outcome, extra=extra, B=B)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · The EU table
One row per EU country. Averages over the years available for each measure.

| Measure | Meaning |
|---|---|
| **Labour cost** | Euro per hour worked in the business economy (wages plus employer social contributions) |
| **Productivity** | Value added per employee, thousand euro |
| **Gross operating rate** | Gross operating surplus (a profit-like measure) as % of turnover, business economy |
| **Profit share** | Gross operating surplus as % of value added, non-financial corporations (national accounts) |
| **ROE** | Net return on equity of non-financial corporations |
| **Min wage (PPS)** | Statutory monthly minimum wage in purchasing power; missing where there is none |

In [2]:
def eu(like, y0, y1, name):
    d = q(f"SELECT iso3, AVG(value) AS v FROM fact_country_year WHERE indicator_id LIKE '{like}' AND year BETWEEN {y0} AND {y1} GROUP BY 1")
    return d.set_index("iso3").v.rename(name)
tbl = pd.concat([
    eu("ESTAT:lc_lci_lev:%", 2020, 2024, "labour cost (€/h)"), eu("ESTAT:sbs_sc_ovw:%AV_SAL_TEUR%", 2021, 2024, "productivity (k€/employee)"),
    eu("ESTAT:sbs_sc_ovw:%GOR_PC%", 2021, 2024, "gross operating rate (%)"), eu("ESTAT:nasa_10_ki:%B2G_B3G_RAT_S11%", 2019, 2023, "profit share NFC (%)"),
    eu("ESTAT:nasa_10_ki:%ROE_S11%", 2019, 2023, "ROE NFC (%)"), eu("ESTAT:earn_mw_cur:currency=PPS", 2022, 2025, "min wage (PPS)"),
    eu("ESTAT:une_rt_a:%Y15-74%", 2019, 2023, "unemployment (%)"), eu("ESTAT:une_rt_a:%Y15-24%", 2019, 2023, "youth unemployment (%)"),
    eu("ESTAT:isoc_eb_ai:%", 2023, 2025, "AI adoption (%)"), eu("ESTAT:rd_e_gerdtot:%PC_GDP%", 2019, 2023, "R&D (% GDP)"),
], axis=1)
firm = fy.groupby("iso3").agg(**{"median ROA (pp)": ("roa_pp", "median"), "median op. margin (%)": ("operating_margin_w", lambda s: s.median() * 100), "loss share (%)": ("loss_pp", "mean"), "firms": ("company_id", "nunique")})
tbl = dim[dim.is_eu][["name", "eu_group"]].join(tbl).join(firm[firm.firms >= MIN_FIRMS].drop(columns="firms"))
print(f"{len(tbl)} EU countries; countries with a value per measure:", tbl.drop(columns=["name", "eu_group"]).notna().sum().to_dict())
tbl.round(2).sort_values("labour cost (€/h)")

21 EU countries; countries with a value per measure: {'labour cost (€/h)': 21, 'productivity (k€/employee)': 21, 'gross operating rate (%)': 21, 'profit share NFC (%)': 21, 'ROE NFC (%)': 21, 'min wage (PPS)': 16, 'unemployment (%)': 21, 'youth unemployment (%)': 21, 'AI adoption (%)': 21, 'R&D (% GDP)': 21, 'median ROA (pp)': 21, 'median op. margin (%)': 21, 'loss share (%)': 21}


,name,eu_group,labour cost (€/h),productivity (k€/employee),gross operating rate (%),profit share NFC (%),ROE NFC (%),min wage (PPS),unemployment (%),youth unemployment (%),AI adoption (%),R&D (% GDP),median ROA (pp),median op. margin (%),loss share (%)
iso3,,,,,,,,,,,,,,,
ROU,Romania,Eastern,9.920,31.550,13.820,53.320,26.970,"1,040.620",5.560,21.640,3.260,0.480,3.350,7.360,18.570
HUN,Hungary,Eastern,11.540,40.740,11.600,44.130,14.570,856.500,3.820,12.220,7.150,1.490,2.780,6.800,25.930
LVA,Latvia,Eastern,12.560,32.770,11.160,40.810,17.010,776.500,7.080,14.000,8.520,0.760,3.170,3.390,27.030
LTU,Lithuania,Eastern,13.200,37.770,11.760,46.690,31.580,"1,079.750",6.960,14.300,11.640,1.060,5.790,8.840,16.090
POL,Poland,Eastern,13.340,45.260,12.010,46.850,24.070,"1,263.620",3.120,10.920,5.980,1.420,4.550,9.710,18.890
GRC,Greece,Southern,14.980,37.080,12.820,40.570,7.950,"1,040.000",14.760,33.820,7.570,1.430,4.190,12.080,10.100
CZE,Czechia,Eastern,16.360,55.260,11.110,45.170,10.720,828.750,2.440,7.380,11.590,1.900,4.250,10.940,8.820
PRT,Portugal,Southern,16.360,42.560,12.180,36.520,8.660,"1,062.250",6.620,20.780,9.340,1.610,3.170,7.530,12.000
EST,Estonia,Eastern,16.680,45.580,9.880,44.590,12.290,769.500,5.920,16.500,14.160,1.730,2.670,6.540,21.430


## 2 · Primary test: labour cost and productivity
Both on a log scale (a 10% higher labour cost goes with how much higher productivity?). The slope is the *elasticity*: 1 would mean productivity rises one-for-one with labour cost.

In [3]:
pr = tbl[["labour cost (€/h)", "productivity (k€/employee)"]].dropna().join(dim[["eu_group", "name"]], rsuffix="_")
x, y = "labour cost (€/h)", "productivity (k€/employee)"
rho, p_rho, n_rho = spearman(pr[x], pr[y]); lo, hi = boot_ci(pr[x], pr[y])
m = smf.ols("np.log(y_) ~ np.log(x_)", pr.rename(columns={x: "x_", y: "y_"})).fit(cov_type="HC3")
el, el_ci = m.params.iloc[1], m.conf_int().iloc[1]
print(f"n = {n_rho} EU countries | Spearman ρ = {rho:+.2f} (95% bootstrap CI {lo:+.2f} to {hi:+.2f}), p = {p_rho:.5f}")
print(f"Elasticity (log-log slope): {el:.2f} (95% CI {el_ci[0]:.2f} to {el_ci[1]:.2f}): a 10% higher labour cost goes with {el * 10:.1f}% higher productivity.")
fig = px.scatter(pr.reset_index(), x=x, y=y, color="eu_group", text="iso3", log_x=True, log_y=True, title=f"Productivity vs labour cost, EU (ρ = {rho:+.2f}, elasticity {el:.2f}, n = {n_rho})")
fig.update_traces(textposition="top center", marker_size=9); fig.update_layout(height=500); fig.show()

n = 21 EU countries | Spearman ρ = +0.90 (95% bootstrap CI +0.71 to +0.98), p = 0.00000
Elasticity (log-log slope): 0.87 (95% CI 0.72 to 1.03): a 10% higher labour cost goes with 8.7% higher productivity.


## 3 · Exploratory: does expensive labour squeeze profit?
Labour cost against six profit and firm outcomes (Spearman, with Benjamini–Hochberg q across the six). These are **exploratory** and not part of the verdict.

In [4]:
outs = ["gross operating rate (%)", "profit share NFC (%)", "ROE NFC (%)", "median ROA (pp)", "median op. margin (%)", "loss share (%)"]
rows = []
for o in outs:
    r_, p_, n_ = spearman(tbl[x], tbl[o]); lo_, hi_ = boot_ci(tbl[x], tbl[o]) if n_ >= 8 else (np.nan, np.nan)
    rows.append({"outcome": o, "ρ with labour cost": r_, "ci_low": lo_, "ci_high": hi_, "p": p_, "n": n_})
sq = pd.DataFrame(rows).set_index("outcome"); sq["q (BH)"] = bh(sq.p)
print(f"{(sq.p < 0.05).sum()} of {len(sq)} with p < 0.05 ({0.05 * len(sq):.1f} expected by chance); {(sq['q (BH)'] < 0.10).sum()} with q < 0.10")
fig = make_subplots(rows=2, cols=3, subplot_titles=[f"{o} (ρ = {sq.loc[o, 'ρ with labour cost']:+.2f}, n = {int(sq.loc[o, 'n'])})" for o in outs], vertical_spacing=0.15)
for k, o in enumerate(outs):
    d = tbl[[x, o]].dropna()
    fig.add_trace(go.Scatter(x=d[x], y=d[o], mode="markers+text", text=d.index, textposition="top center", textfont_size=8, marker=dict(size=7, color="#4C78A8"), showlegend=False), row=k // 3 + 1, col=k % 3 + 1)
fig.update_layout(height=640, title="Labour cost vs profitability, EU countries"); fig.update_xaxes(title_text="labour cost (€/h)", row=2); fig.show()
sq

0 of 6 with p < 0.05 (0.3 expected by chance); 0 with q < 0.10


,ρ with labour cost,ci_low,ci_high,p,n,q (BH)
outcome,,,,,,
gross operating rate (%),-0.214,-0.625,0.250,0.351,21,0.526
profit share NFC (%),-0.368,-0.692,0.061,0.101,21,0.304
ROE NFC (%),-0.174,-0.614,0.366,0.451,21,0.541
median ROA (pp),-0.238,-0.574,0.172,0.300,21,0.526
median op. margin (%),0.388,-0.057,0.727,0.082,21,0.304
loss share (%),-0.009,-0.482,0.501,0.969,21,0.969


### Minimum wages and unemployment
Minimum wage (PPS) against unemployment and youth unemployment. Five EU countries (Austria, Denmark, Finland, Italy, Sweden) have no statutory minimum, so n is smaller.

In [5]:
rows = []
for o in ["unemployment (%)", "youth unemployment (%)"]:
    r_, p_, n_ = spearman(tbl["min wage (PPS)"], tbl[o]); lo_, hi_ = boot_ci(tbl["min wage (PPS)"], tbl[o])
    rows.append({"outcome": o, "ρ with minimum wage (PPS)": r_, "ci_low": lo_, "ci_high": hi_, "p": p_, "n": n_})
pd.DataFrame(rows).set_index("outcome")

,ρ with minimum wage (PPS),ci_low,ci_high,p,n
outcome,,,,,
unemployment (%),-0.159,-0.641,0.352,0.557,16
youth unemployment (%),-0.121,-0.628,0.393,0.656,16


### EU groups
Nordic, Western, Southern, Eastern on labour cost, productivity and gross operating rate (Kruskal–Wallis across groups).

In [6]:
cols = ["labour cost (€/h)", "productivity (k€/employee)", "gross operating rate (%)"]
fig = make_subplots(rows=1, cols=3, subplot_titles=cols)
colors = {"Nordic": "#4C78A8", "Western": "#F58518", "Southern": "#E45756", "Eastern": "#54A24B"}
for k, c in enumerate(cols, 1):
    for g, gg in tbl.groupby("eu_group"):
        fig.add_trace(go.Box(y=gg[c], name=g, boxpoints="all", text=gg.name, jitter=.5, marker_color=colors[g], showlegend=False), row=1, col=k)
fig.update_layout(height=420); fig.show()
pd.Series({c: stats.kruskal(*[g[c].dropna().values for _, g in tbl.groupby("eu_group")]).pvalue for c in cols}, name="Kruskal–Wallis p")

labour cost (€/h)            0.001
productivity (k€/employee)   0.002
gross operating rate (%)     0.438
Name: Kruskal–Wallis p, dtype: float64

## 4 · Robustness
The primary link without each country in turn, without the two high-productivity outliers (Ireland, Luxembourg), and with the euro area only. The robustness share is the share of variants with a positive ρ.

In [7]:
vv = {}
for iso in pr.index:
    vv[f"leave out {iso}"] = spearman(pr.drop(index=iso)[x], pr.drop(index=iso)[y])[0]
ex = pr.drop(index=[i for i in ["IRL", "LUX"] if i in pr.index]); vv["without Ireland, Luxembourg"] = spearman(ex[x], ex[y])[0]
ez = pr[pr.index.isin(dim.index[dim.is_euro])]; vv["euro area only"] = spearman(ez[x], ez[y])[0]
nz = pr[~pr.index.isin(dim.index[dim.is_euro])]; vv["non-euro only"] = spearman(nz[x], nz[y])[0]
vs = pd.Series(vv); rob_share = float((vs > 0).mean())
print(f"Positive ρ in {rob_share:.0%} of {len(vs)} variants; range {vs.min():+.2f} to {vs.max():+.2f}.")
vs.rename("ρ").to_frame().T.round(2)

Positive ρ in 96% of 24 variants; range +0.87 to +0.95.


,leave out AUT,leave out BEL,leave out CZE,leave out DNK,leave out EST,leave out FIN,leave out FRA,leave out DEU,leave out GRC,leave out HUN,leave out IRL,leave out ITA,leave out LVA,leave out LTU,leave out LUX,leave out NLD,leave out POL,leave out PRT,leave out ROU,leave out ESP,leave out SWE,"without Ireland, Luxembourg",euro area only,non-euro only
ρ,0.910,0.900,0.890,0.900,0.890,0.900,0.920,0.920,0.900,0.900,0.950,0.890,0.890,0.890,0.900,0.900,0.890,0.890,0.890,0.890,0.920,0.940,0.870,NaN


## 5 · Replication with another measure
Output per person from the World Bank (GDP per person at purchasing power, 2019–2023) instead of Eurostat's value added per employee, against the same Eurostat labour cost.

In [8]:
alt = tbl[[x]].join(profile_all[["gdp_per_capita_ppp"]]).dropna()
r_alt, p_alt, n_alt = spearman(alt[x], alt.gdp_per_capita_ppp)
print(f"Labour cost vs World Bank GDP per person (PPP): ρ = {r_alt:+.2f}, p = {p_alt:.4f}, n = {n_alt}")

Labour cost vs World Bank GDP per person (PPP): ρ = +0.88, p = 0.0000, n = 21


## 6 · Verdict

In [9]:
v = utils.verdict(rho, p_rho, +1, rob_share, int(n_rho))
lines = [
 f"Primary: labour cost vs productivity ρ = {rho:+.2f} (95% CI {lo:+.2f} to {hi:+.2f}), p = {p_rho:.5f}, n = {n_rho} EU countries; elasticity {el:.2f} (a 10% higher labour cost goes with {el * 10:.1f}% higher productivity).",
 f"Robustness: positive in {rob_share:.0%} of {len(vs)} variants. Alternative output measure (World Bank GDP per person): ρ = {r_alt:+.2f}.",
 f"Exploratory: labour cost vs profit measures: " + "; ".join(f"{o.split(' (')[0]} {sq.loc[o, 'ρ with labour cost']:+.2f}" for o in outs) + f". {(sq['q (BH)'] < 0.10).sum()} of {len(sq)} have q < 0.10.",
 f"VERDICT H6: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H6", id="H6", title="EU labour cost and productivity", statement="Higher hourly labour costs go with higher productivity in the EU.",
                  expected_sign=1, effect=float(rho), ci_low=float(lo), ci_high=float(hi), effect_unit="Spearman ρ, hourly labour cost vs value added per employee",
                  p_primary=float(p_rho), p_method="Spearman test (analytic)", n=int(n_rho), clusters=int(n_rho), robust_share=rob_share,
                  replication_coef=float(r_alt), replication_p=float(p_alt), replication_same_sign=bool(r_alt > 0), verdict=v,
                  elasticity=float(el), notebook="1.6_eu_labour_cost_and_productivity.ipynb", notes=lines)

Primary: labour cost vs productivity ρ = +0.90 (95% CI +0.71 to +0.98), p = 0.00000, n = 21 EU countries; elasticity 0.87 (a 10% higher labour cost goes with 8.7% higher productivity).
Robustness: positive in 96% of 24 variants. Alternative output measure (World Bank GDP per person): ρ = +0.88.
Exploratory: labour cost vs profit measures: gross operating rate -0.21; profit share NFC -0.37; ROE NFC -0.17; median ROA -0.24; median op. margin +0.39; loss share -0.01. 0 of 6 have q < 0.10.
VERDICT H6: SUPPORTED


### Limits
- 21 countries; the EU is a narrower, richer group than the 34, so this says little about the rest of the world.
- Labour cost (per hour) and productivity (per employee) use different bases (hours vs heads), and differences in part-time work shift the comparison.
- Eurostat's gross operating rate and the national-accounts profit share are *not* the same thing as company ROA; different definitions, different coverage (all firms vs listed firms).
- The profit section is exploratory: six tests, small n.
- Association, not causation.

In [10]:
con.close()